# 🎙️ Deepfake Voice Detection — Wav2Vec2 Embedding Extraction
### Direct High-Speed Download (No Keys / No Login Required) → Colab T4 GPU

**What this notebook does:**
1. **Directly downloads** the official ASVspoof 2019 Logical Access (`LA.zip`) dataset via Edinburgh DataShare (~7 GB at ~80 MB/s in ~1-2 min)
2. Extracts 768-dim Wav2Vec2 embeddings using a **free T4 GPU**
3. Generates `dev_embeddings.pt` and `train_embeddings.pt`
4. Automatically downloads them to your local PC

**⏱️ Total time:** ~20–25 minutes

---
### ✅ Before running:
- Ensure GPU is enabled: **Runtime → Change runtime type → T4 GPU**

---
## Step 1 — Verify GPU

In [ ]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {device}')
if device == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    print('✅ GPU ready!')
else:
    print('⚠️ No GPU detected!')
    print('   Please go to: Runtime → Change runtime type → Hardware accelerator → T4 GPU')
    print('   Then re-run this cell.')

---
## Step 2 — Direct Dataset Download (No Login Needed)

Downloads `LA.zip` directly from Edinburgh DataShare servers into Colab storage.

In [ ]:
import os
import time

print('⬇️ Downloading ASVspoof 2019 LA dataset directly from Edinburgh DataShare...')
t0 = time.time()

# Direct download (supports resuming with -c)
!wget -c https://datashare.ed.ac.uk/bitstream/handle/10283/3336/LA.zip -O /content/LA.zip

print(f'Download finished in {(time.time() - t0)/60:.1f} minutes. Unzipping...')
t1 = time.time()
!unzip -q /content/LA.zip -d /content/
print(f'✅ Unzip complete in {(time.time() - t1)/60:.1f} minutes!')

---
## Step 3 — Verify Dataset Structure

In [ ]:
from pathlib import Path

# Check standard unzipped location
candidates = [
    Path('/content/LA'),
    Path('/content/LA/LA'),
]
DATA_ROOT = None
for c in candidates:
    if (c / 'ASVspoof2019_LA_cm_protocols').exists():
        DATA_ROOT = c
        break

if DATA_ROOT is None:
    # Fallback search
    import glob
    matches = list(Path('/content').rglob('ASVspoof2019_LA_cm_protocols'))
    if matches:
        DATA_ROOT = matches[0].parent

if DATA_ROOT is None:
    raise RuntimeError('Could not find ASVspoof2019_LA_cm_protocols directory!')

print(f'✅ Dataset located at: {DATA_ROOT}\n')

required = [
    'ASVspoof2019_LA_train/flac',
    'ASVspoof2019_LA_dev/flac',
    'ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt',
    'ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.dev.trl.txt',
]

for r in required:
    p = DATA_ROOT / r
    print(f'  {"✅" if p.exists() else "❌"} {r}')

n_train = len(list((DATA_ROOT / 'ASVspoof2019_LA_train/flac').glob('*.flac')))
n_dev   = len(list((DATA_ROOT / 'ASVspoof2019_LA_dev/flac').glob('*.flac')))
print(f'\n  Found {n_train:,} train clips and {n_dev:,} dev clips.')

---
## Step 4 — Install Dependencies & Load Wav2Vec2

In [ ]:
%%capture
!pip install -q transformers torchaudio soundfile

import torchaudio
import transformers
from transformers import Wav2Vec2FeatureExtractor, Wav2Vec2Model

MODEL_NAME = 'facebook/wav2vec2-base'
TARGET_SR  = 16_000

print(f'Loading {MODEL_NAME} onto {device}...')
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_NAME)
model = Wav2Vec2Model.from_pretrained(MODEL_NAME)
model.eval().to(device)
print(f'✅ Model ready on {device}!')

---
## Step 5 — Helper Functions

In [ ]:
import time
import numpy as np

SPLIT_CONFIG = {
    'train': {
        'audio_dir' : 'ASVspoof2019_LA_train/flac',
        'protocol'  : 'ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt',
    },
    'dev': {
        'audio_dir' : 'ASVspoof2019_LA_dev/flac',
        'protocol'  : 'ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.dev.trl.txt',
    },
    'eval': {
        'audio_dir' : 'ASVspoof2019_LA_eval/flac',
        'protocol'  : 'ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.eval.trl.txt',
    },
}
LABEL_MAP = {'bonafide': 0, 'spoof': 1}


def parse_protocol(path):
    entries = []
    with open(path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 5:
                entries.append((parts[1], parts[-1]))
    return entries


def load_audio(path):
    wav, sr = torchaudio.load(str(path))
    if wav.shape[0] > 1:
        wav = wav.mean(0, keepdim=True)
    if sr != TARGET_SR:
        wav = torchaudio.functional.resample(wav, sr, TARGET_SR)
    return wav.squeeze(0)


def extract_split(split_name, limit=None):
    cfg       = SPLIT_CONFIG[split_name]
    audio_dir = DATA_ROOT / cfg['audio_dir']
    proto     = DATA_ROOT / cfg['protocol']

    entries = parse_protocol(proto)
    if limit:
        entries = entries[:limit]
    total = len(entries)

    print(f'\n━━━ [{split_name.upper()}] Extracting {total:,} clips ━━━', flush=True)

    embeddings, labels, filenames = [], [], []
    skipped = 0
    t0 = time.time()

    with torch.no_grad():
        for i, (fname, label_str) in enumerate(entries):
            flac = audio_dir / f'{fname}.flac'
            if not flac.exists():
                skipped += 1
                continue

            wav    = load_audio(flac)
            inputs = feature_extractor(
                wav.numpy(), sampling_rate=TARGET_SR, return_tensors='pt'
            )
            out = model(inputs.input_values.to(device))
            # Mean pool across time dimension -> 768-dim vector
            emb = out.last_hidden_state.mean(dim=1).squeeze(0).cpu()

            embeddings.append(emb)
            labels.append(LABEL_MAP[label_str])
            filenames.append(fname)

            if (i + 1) % 500 == 0 or (i + 1) == total:
                done    = i + 1 - skipped
                elapsed = time.time() - t0
                rate    = done / elapsed
                eta_min = (total - i - 1) / rate / 60
                print(f'  {i+1:>6}/{total}  |  {rate:.1f} clips/s  |  ETA ~{eta_min:.1f} min', flush=True)

    if skipped:
        print(f'  ⚠️ Skipped {skipped} missing files')

    E = torch.stack(embeddings)
    L = torch.tensor(labels, dtype=torch.long)
    print(f'  Output shape : {tuple(E.shape)}')
    print(f'  Class counts : bonafide={int((L==0).sum())} | spoof={int((L==1).sum())}')
    return {'embeddings': E, 'labels': L, 'filenames': filenames}


print('✅ Helper functions ready.')

---
## Step 6 — Extract Dev Embeddings (~5 min)

In [ ]:
OUTPUT_DIR = Path('/content/embeddings')
OUTPUT_DIR.mkdir(exist_ok=True)

dev_data = extract_split('dev')

dev_out = OUTPUT_DIR / 'dev_embeddings.pt'
torch.save(dev_data, dev_out)
print(f'\n✅ Saved → {dev_out}  ({dev_out.stat().st_size/1e6:.0f} MB)')

---
## Step 7 — Extract Train Embeddings (~15 min)

In [ ]:
train_data = extract_split('train')

train_out = OUTPUT_DIR / 'train_embeddings.pt'
torch.save(train_data, train_out)
print(f'\n✅ Saved → {train_out}  ({train_out.stat().st_size/1e6:.0f} MB)')

---
## Step 8 — Download `.pt` Files to Your PC

Runs browser downloads for both `train_embeddings.pt` and `dev_embeddings.pt`.

In [ ]:
from google.colab import files

print('📦 Embedding files generated:')
for fname in ['train_embeddings.pt', 'dev_embeddings.pt']:
    fpath = OUTPUT_DIR / fname
    if fpath.exists():
        size_mb = fpath.stat().st_size / 1e6
        print(f'  ✅ {fname} ({size_mb:.0f} MB)')

print('\n📂 Move these downloaded files into your local project at:')
print('   c:\\Deepfake-voice-detection\\data\\embeddings\\\n')

# Trigger browser download
for fname in ['train_embeddings.pt', 'dev_embeddings.pt']:
    fpath = OUTPUT_DIR / fname
    if fpath.exists():
        print(f'⬇️ Downloading {fname} to your browser...')
        files.download(str(fpath))